<a href="https://colab.research.google.com/github/ryueuitae/core-study/blob/main/chapter_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 9. 비지도 학습

레이블(정답) 없이 데이터 자체의 구조를 찾는 학습

* 실제 데이터는 대부분 레이블이 없음 → 레이블을 다는 데 시간과 비용이 큼
* 대표 작업
  * 군집(clustering): 비슷한 샘플끼리 그룹으로 묶기
  * 이상치 탐지(anomaly detection): 정상 데이터와 다른 샘플 찾기
  * 밀도 추정(density estimation): 데이터가 어떤 확률 분포에서 나왔는지 추정

# 군집(Clustering)
비슷한 샘플을 구별해 하나의 클러스터(그룹)로 할당하는 작업

활용 예시
* 고객 분류: 구매 패턴이 비슷한 고객끼리 묶어 추천에 활용
* 데이터 분석: 데이터를 클러스터별로 나눠서 따로 분석
* 차원 축소: 각 샘플을 클러스터와의 친밀도로 표현
* 이상치 탐지: 어떤 클러스터에도 잘 안 맞는 샘플 = 이상치
* 준지도 학습: 레이블이 몇 개 없을 때 같은 클러스터로 레이블 전파
* 이미지 분할: 비슷한 색의 픽셀끼리 묶기

## 1. k-평균(k-means)
데이터를 k개의 클러스터로 나누는 가장 기본적인 군집 알고리즘

알고리즘
1. 센트로이드(클러스터 중심) k개를 무작위로 놓음
2. 각 샘플을 가장 가까운 센트로이드의 클러스터에 할당
3. 각 클러스터에 속한 샘플들의 평균으로 센트로이드를 이동
4. 센트로이드가 더 이상 움직이지 않을 때까지 2~3 반복

* 클러스터 개수 k는 사람이 미리 정해야 함
* 하드 군집: 샘플마다 클러스터 하나에만 할당
* 소프트 군집: 샘플마다 각 클러스터와의 거리(점수)를 줌 → `transform()`

In [1]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans

blob_centers = np.array([[0.2, 2.3], [-1.5, 2.3], [-2.8, 1.8], [-2.8, 2.8], [-2.8, 1.3]])
blob_std = np.array([0.4, 0.3, 0.1, 0.1, 0.1])
X, y = make_blobs(n_samples=2000, centers=blob_centers, cluster_std=blob_std, random_state=7)

kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
y_pred = kmeans.fit_predict(X)     # 각 샘플이 몇 번 클러스터인지
print(kmeans.cluster_centers_)     # 찾은 센트로이드 5개

[[-2.80214068  1.55162671]
 [ 0.08703534  2.58438091]
 [-1.46869323  2.28214236]
 [-2.79290307  2.79641063]
 [ 0.31332823  1.96822352]]


### 센트로이드 초기화와 속도 개선
* 문제: 처음 센트로이드를 어디 놓느냐에 따라 결과가 달라지고, 나쁜 해(지역 최적점)에 빠질 수 있음
* 해결 1) 여러 번 실행해서 가장 좋은 결과 선택 (`n_init`)
  * 좋은 결과의 기준 = 이너셔(inertia): 각 샘플과 가장 가까운 센트로이드 사이 거리 제곱의 합, 작을수록 좋음
* 해결 2) k-평균++: 센트로이드끼리 멀리 떨어지도록 똑똑하게 초기화 (사이킷런 기본값)
* 가속 k-평균: 불필요한 거리 계산을 건너뛰어 속도 향상 (`algorithm="elkan"`)
* 미니배치 k-평균: 전체 데이터 대신 작은 배치로 센트로이드를 조금씩 이동 → 큰 데이터에 유리 (`MiniBatchKMeans`)

### 최적의 클러스터 개수 찾기
방법 1) 이너셔 그래프의 엘보
* k를 늘리면 이너셔는 항상 줄어듦 → 이너셔만으로는 k를 정할 수 없음
* 이너셔가 급격히 줄다가 완만해지는 지점(엘보)을 보고 판단 → 대략적인 방법

방법 2) 실루엣 점수 (더 정확)
* 각 샘플이 자기 클러스터에 얼마나 잘 속해 있고, 다른 클러스터와 얼마나 떨어져 있는지 측정
* -1 ~ 1 사이 값
  * 1에 가까움: 자기 클러스터에 잘 속하고 다른 클러스터와 멀리 떨어짐
  * 0 근처: 클러스터 경계에 있음
  * -1에 가까움: 잘못된 클러스터에 할당됐을 가능성
* 모든 샘플의 평균 실루엣 점수가 가장 높은 k 선택

In [2]:
from sklearn.metrics import silhouette_score

for k in range(2, 8):
    kmeans_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    print(k, silhouette_score(X, kmeans_k.labels_))

2 0.5966442557582528
3 0.5723900247411775
4 0.688531617595759
5 0.6353422668284152
6 0.6020248775444942
7 0.6073764342983768


### 군집 활용 예시
이미지 분할
* 픽셀의 색(RGB)을 k-평균으로 군집 → 각 픽셀을 클러스터 평균 색으로 바꿈
* k가 작을수록 색이 단순해짐

준지도 학습
* 레이블이 적을 때: 데이터를 k개로 군집하고 각 센트로이드에 가장 가까운 대표 샘플에만 레이블을 붙임
* 레이블 전파: 대표 샘플의 레이블을 같은 클러스터의 다른 샘플에도 복사 → 적은 레이블로 성능 크게 향상
* 능동 학습: 모델이 가장 헷갈려하는 샘플을 사람에게 물어서 레이블을 추가하는 방식

## 2. DBSCAN
밀도가 높은 지역을 하나의 클러스터로 보는 군집 알고리즘

* 각 샘플의 반경 ε(eps) 안에 이웃이 몇 개 있는지 셈
* 이웃이 min_samples개 이상이면 핵심 샘플
* 핵심 샘플끼리 이웃이면 같은 클러스터로 연결
* 핵심 샘플도 아니고 핵심 샘플의 이웃도 아니면 이상치 → 레이블 -1

장점
* 클러스터 개수를 미리 정할 필요 없음
* 클러스터 모양에 상관없이 찾을 수 있음 (초승달 모양 등)
* 이상치를 자동으로 구분

한계
* 새 샘플을 예측하는 `predict()`가 없음 → KNN 분류기 등을 따로 학습해서 사용
* 클러스터마다 밀도가 크게 다르면 eps 하나로 잡기 어려움

In [3]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

X_moons, y_moons = make_moons(n_samples=1000, noise=0.05, random_state=42)
dbscan = DBSCAN(eps=0.2, min_samples=5)
dbscan.fit(X_moons)
print(set(dbscan.labels_))   # 찾은 클러스터 번호 (-1은 이상치)

{np.int64(0), np.int64(1)}


## 3. 다른 군집 알고리즘
* 병합 군집: 가까운 클러스터끼리 아래에서부터 하나씩 합쳐 나감 → 트리 구조
* BIRCH: 대용량 데이터용, 트리 구조로 빠르게 군집
* 평균-이동: 각 샘플 주변 밀도가 높은 방향으로 이동 → 모인 곳이 클러스터
* 유사도 전파: 샘플끼리 메시지를 주고받으며 대표 샘플을 선출
* 스펙트럼 군집: 샘플 간 유사도 행렬을 저차원으로 줄인 뒤 군집

# 가우스 혼합(Gaussian Mixture, GMM)
데이터가 여러 개의 가우스 분포(정규분포)가 섞여서 만들어졌다고 가정하는 확률 모델

* 각 클러스터 = 타원 모양의 가우스 분포 하나 (크기, 모양, 방향이 제각각 가능)
* k-평균의 일반화: k-평균은 원형 클러스터만, GMM은 타원형 클러스터도 가능
* 소프트 군집: 각 샘플이 각 클러스터에 속할 확률을 줌 (`predict_proba()`)

EM 알고리즘으로 학습
* E 단계: 현재 분포로 각 샘플이 각 클러스터에 속할 확률 계산
* M 단계: 그 확률을 가중치로 분포의 평균, 공분산, 비중 업데이트
* 수렴할 때까지 반복 → k-평균의 할당/업데이트 반복과 비슷한 구조

생성 모델
* 학습한 분포에서 새로운 샘플을 만들 수 있음 (`sample()`)
* 각 위치의 밀도를 계산할 수 있음 (`score_samples()`)

In [4]:
from sklearn.mixture import GaussianMixture

gm = GaussianMixture(n_components=5, n_init=10, random_state=42)
gm.fit(X)
print(gm.weights_)             # 각 클러스터의 비중
print(gm.predict_proba(X[:3])) # 처음 3개 샘플이 각 클러스터에 속할 확률

[0.08904483 0.4001636  0.19975271 0.19986052 0.11117835]
[[2.60774919e-017 9.99999366e-001 9.17262145e-048 6.34452771e-007
  2.43246809e-016]
 [5.73846669e-017 9.99998438e-001 1.27356117e-022 1.56185706e-006
  2.57546267e-019]
 [6.16830145e-007 8.67135424e-296 0.00000000e+000 8.02256350e-015
  9.99999383e-001]]


### 가우스 혼합을 이용한 이상치 탐지
* 밀도가 낮은 지역에 있는 샘플 = 이상치
* 밀도 임계값을 정함 (예: 밀도 하위 2%를 이상치로 판단)
* 이상치 탐지: 훈련 데이터에 이상치가 섞여 있을 수 있음
* 특이치 탐지(novelty detection): 깨끗한 데이터로 학습하고, 새로 들어온 샘플이 이상한지 판단

### 클러스터 개수 선택
* GMM은 원형이 아닐 수 있어서 이너셔, 실루엣 점수가 잘 안 맞음
* 대신 BIC 또는 AIC가 최소가 되는 클러스터 수 선택
  * 모델이 데이터를 잘 설명할수록 낮아짐
  * 파라미터(클러스터)가 많을수록 벌점 → 너무 많은 클러스터 방지

베이즈 가우스 혼합 (`BayesianGaussianMixture`)
* 클러스터 수를 넉넉하게 주면 불필요한 클러스터의 비중을 0에 가깝게 만들어 알아서 개수를 정함

# 이상치/특이치 탐지를 위한 다른 알고리즘
* Fast-MCD: 정상 데이터가 하나의 가우스 분포라고 보고, 이상치의 영향을 줄여서 분포 추정 (`EllipticEnvelope`)
* 아이솔레이션 포레스트: 무작위로 데이터를 계속 나눴을 때 빨리 고립되는 샘플 = 이상치, 고차원 데이터에 효율적
* LOF(지역 이상치 인자): 주변 이웃보다 밀도가 훨씬 낮은 샘플 = 이상치
* one-class SVM: 정상 데이터를 감싸는 경계를 학습, 특이치 탐지에 적합
* PCA: 재구성 오차가 큰 샘플 = 이상치 (8장 압축에서 본 재구성 오차 활용)

# 정리
| 기법 | 핵심 | 장점 | 한계 |
|---|---|---|---|
| k-평균 | 센트로이드에 가까운 샘플끼리 묶음 | 빠르고 단순 | k 지정 필요, 원형 클러스터만 |
| DBSCAN | 밀도 높은 지역을 클러스터로 | 모양 무관, 이상치 자동 구분 | predict 없음, 밀도 차이에 약함 |
| 가우스 혼합 | 여러 가우스 분포의 혼합으로 모델링 | 타원형 클러스터, 확률, 이상치 탐지 | 차원 높으면 느림 |